# 重要な文を選んで抽出型要約を作る (LexRank)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_topics_keywords/extractive_summary.ipynb)

NLP クックブック 12-05 ／ トピック・キーワード・要約

長い文章から、内容を代表する文をいくつか抜き出して要約にしたい。生成モデルと違い、原文にない内容が混ざらない。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import re
import networkx as nx
import numpy as np
from fugashi import Tagger
from sklearn.feature_extraction.text import TfidfVectorizer

tagger = Tagger()
tok = lambda s: [w.feature.orthBase or w.surface for w in tagger(s) if w.feature.pos1 in ("名詞", "動詞", "形容詞") and w.feature.pos2 != "非自立可能"]

def lexrank(text, n=3, threshold=0.1):
    sents = [s.strip() for s in re.split(r"(?<=[。！？])", text) if s.strip()]
    X = TfidfVectorizer(tokenizer=tok, token_pattern=None, lowercase=False).fit_transform(sents)
    sim = (X @ X.T).toarray()                      # 文同士のコサイン類似度
    np.fill_diagonal(sim, 0)
    sim[sim < threshold] = 0
    scores = nx.pagerank(nx.from_numpy_array(sim), weight="weight")   # 多くの文と似ている文 = 中心的な文
    top = sorted(sorted(scores, key=scores.get, reverse=True)[:n])    # スコア上位を、元の順番で並べる
    return [sents[i] for i in top], [round(scores[i], 3) for i in range(len(sents))]

text = ("自然言語処理は、人間が日常的に使っている言語をコンピュータに処理させる技術である。"
        "その歴史は 1950 年代の機械翻訳の研究にさかのぼる。"
        "初期の機械翻訳は人手で書かれた規則に基づいていたが、十分な精度は得られなかった。"
        "1990 年代になると、大量の対訳データから統計的に翻訳を学習する統計的機械翻訳が主流となった。"
        "2010 年代半ばには、ニューラルネットワークを用いたニューラル機械翻訳が登場し、翻訳の精度は飛躍的に向上した。"
        "現在では、機械翻訳に限らず、自然言語処理のほとんどのタスクでニューラルネットワークが使われている。"
        "特に、大量のテキストで事前学習された大規模言語モデルは、少ない追加学習で多様なタスクに適用できる。"
        "ところで、昨日の夕食はカレーだった。")
summary, scores = lexrank(text)
print(scores)
for s in summary:
    print("・", s)

[0.109, 0.095, 0.11, 0.167, 0.205, 0.206, 0.087, 0.021]
・ 1990 年代になると、大量の対訳データから統計的に翻訳を学習する統計的機械翻訳が主流となった。
・ 2010 年代半ばには、ニューラルネットワークを用いたニューラル機械翻訳が登場し、翻訳の精度は飛躍的に向上した。
・ 現在では、機械翻訳に限らず、自然言語処理のほとんどのタスクでニューラルネットワークが使われている。


- 文をノード、類似度を辺の重みとするグラフを作り、PageRank で中心的な文を選ぶ。無関係な文 (最後の文) はどの文とも似ていないのでスコアが低い。
- 類似度の計算を TF-IDF から文埋め込みに替えると、言い換えにも強くなる。
- ニュース記事では「先頭の 3 文を取る」(Lead-3) が非常に強いベースラインになる。要約手法を評価するときは必ず比較する。
- 英語を含む各種の抽出型要約アルゴリズムは `sumy` にまとまっている。